# Absorption of Conjugated Dyes with the Finite Well


<div>
  <center> <img height="500" src="https://github.com/pmpatel-udallas/CHE-3131/blob/main/Rotation-4-Quantum-Theory/Figures/Fig6_UVvis.png?raw=True"> </center>
</div>


<a target="_blank" href="https://colab.research.google.com/github/pmpatel-udallas/CHE-3131/blob/main/Rotation-4-Quantum-Theory/Quantum_Theory.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


This notebook was adapted for the following publication:

* Patel, P. Modernizing Physical Chemistry: Integrating Computational Chemistry, the Finite Well, and Python Data Visualization in the Particle-in-a-Box Experiment. In *Engaging Students in Physical Chemistry, Volume 2; ACS Symposium Series*; American Chemical Society, **2025**; Vol. 1515, pp 261–278. https://doi.org/doi:10.1021/bk-2025-1515.ch017.

In [ ]:
# @title Overview
%%html
<style>
div.alert {
    color: #0056b3;
    background-color: #d9edf7;
    border-left: 5px solid #31708f;
    padding: 0.5em;
    font-size: 1.25em;
    line-height: 1.5;
}
div.alert ul {
    margin: 0.5em 0;
}
div.alert li {
    margin-bottom: 0.5em;
}
</style>

 <head>
    <!-- See http://docs.mathjax.org/en/latest/web/start.html#using-mathjax-from-a-content-delivery-network-cdn -->
    <script type="text/javascript" id="MathJax-script" async
        src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js">
    </script>
</head>

<div class="alert alert-block alert-info">
    <strong>Science Questions:</strong>
    <ul>
        <li>How does the particle-in-a-box (PIB) model connect to UV-vis spectroscopy?</li>
        <li>How does this connection extend to the finite well model?</li>
    </ul>

    <strong>Programming Objectives:</strong>
    <ul>
        <li>Use a for loop to plot the computed UV-vis spectra</li>
        <li>Overlay the computed and experimental spectra</li>
        <li>Determine the computed \( \lambda_{max}\) from the various models and spectra</li>
    </ul>
</div>



---

## Quantum Background

### Infinite Well
In the particle-in-a-box (PIB) model, a particle is confined to moving in the x-dimension of a box of length L. The potential, $V(x)$, is zero inside the box ($0 ≤ x ≤ L$) while $V(x)$ is infinite outside the box ($x ≤ 0$ and $x ≥ L$). Confining the particle inside a one-dimensional (1D) box leads to a quantization of allowed energies. Outside the box, $V(x)$ is infinite and thus, $\Psi(x)$ is zero. In quantum mechanical terms, $\Psi^2(x)$ being zero outside the box means there is zero-probability of finding the particle outside the box.

<div>
  <center> <img height="300" src="https://github.com/pmpatel-udallas/CHE-3131/blob/main/Rotation-4-Quantum-Theory/Figures/Fig1_PIB_Infinite.png?raw=True"> </center>
</div>

Far from being only a pedagogical and mathematical exercise, confining a particle to moving in a 1D box has applications in predicting properties of dye molecules that can be measured. The energy levels predicted by solutions to the 1D PIB problem may be used to predict the maximum absorption wavelength, $\lambda_{max}$, for some chemical compounds that absorb light in the visible spectrum. Such compounds generally have delocalized electrons from either the $\pi$ electrons in a conjugated organic molecule or the odd electron in a free radical species.

---

### Computational Chemistry
Computational chemistry has advanced with the digital age, offering critical insight into chemical processes and properties that are difficult to measure experimentally. Beyond predictive capabilities, computational chemistry can also reinforce the conceptual foundations of the PIB model. For example, consider the figure below, which shows how the 1D PIB framework maps onto (3E, 5E, 7E)-1,3,5,7,9-decapentaene. Computed molecular orbitals (MOs) visually encode the same nodal behavior and energy spacing consistent with the PIB model, illustrating how computational tools can bridge fundamental quantum models with observable molecular behavior.

<div>
  <center> <img height="500" src="https://github.com/pmpatel-udallas/CHE-3131/blob/main/Rotation-4-Quantum-Theory/Figures/Fig2_PIB.png?raw=True"> </center>
</div>


In [ ]:
# @title Hit the play button to import packages

%%capture

#Import standard packages
import numpy as np
import os,sys,re # Import regex
import pandas as pd # DataFrame analysis

# Plotting
import matplotlib
matplotlib.rcParams.update({'font.size': 20})
import matplotlib.pyplot as plt

#
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from mpl_toolkits.mplot3d import Axes3D # 3D plots
from matplotlib import cm # Colormaps
from matplotlib.colors import ListedColormap, LinearSegmentedColormap
from mpl_toolkits.axes_grid1 import make_axes_locatable

#Inset figures into plots
from matplotlib.offsetbox import TextArea, DrawingArea, OffsetImage, AnnotationBbox
import matplotlib.image as mpimg

#Create lines for custom legends
import matplotlib.lines as mlines
from matplotlib.lines import Line2D

# Numerical solver in scipy
import scipy
from scipy.optimize import fsolve

from glob import glob
import warnings

# Insert a progress bar to show the progress of the script
!jupyter nbextension enable --py widgetsnbextension
from tqdm.notebook import tqdm, tnrange, trange

# Py3Dmol for visualizing molecules
!pip install py3Dmol
import py3Dmol

#Quizzes and Animations
from IPython.display import display, HTML, clear_output
import time

%pip install "jupyterquiz"
from jupyterquiz import display_quiz

#Import quizzes and rotation4.json from GitHub
!wget https://raw.githubusercontent.com/pmpatel-udallas/CHE-3131/main/Rotation-4-Quantum-Theory/Broadening.py
!wget https://raw.githubusercontent.com/pmpatel-udallas/CHE-3131/main/Rotation-3-Thermodynamics/FiniteWell.py
!wget https://raw.githubusercontent.com/pmpatel-udallas/CHE-3131/main/quiz_utils.py

import quiz_utils

# Part 1. Combining Experimental and Computational Spectra

## 1.1 Import your experimental and computational data

The experimental data from the UV-vis will be in a csv format, so use `pd.read_csv()` instead of `pd.read_excel()`.

In [ ]:
# Import your experimental data


# Import your computational data



## 1.2 Converting the computed values to a spectrum

The extracted results from the ORCA calculations contain information about the excitation wavelength ($\lambda$) and the oscillator strength ($f$). You will need to broaden each transition using

$$\varepsilon_i(\lambda)=1.3062974*10^8\dfrac{f_i}{\sigma} \text{exp}\bigg[-\bigg(\dfrac{1/\lambda-1/\lambda_i}{\sigma}\bigg)^2\bigg] \tag{1}$$

where $i$ represents the $i^\text{th}$ electronic excitation, $\lambda$ is the excitation wavelength in nm, $f_i$ is the oscillator strength (a.u.), and $\sigma$ is the parameter for peak broadening. Setting $\sigma$ = 0.2 eV = 1/6199.2 nm$^{-1}$ = 10$^{7}$/6199.2 cm$^{-1}$ will result in the following reduction

$$	\varepsilon_i(\lambda)=1.3062974*10^8\dfrac{f_i}{10^{7}/6199.2} \text{exp}\bigg[-\bigg(\dfrac{1/\lambda-1/\lambda_i}{1/6199.2}\bigg)^2\bigg] \tag{2}
$$

Equation 2 above refers to how to broaden a single transition whereas the next equation covers how to do the sum of all the individual excitations.

$$
	\varepsilon(\lambda)=\sum_{i=1}^{n}\varepsilon_i(\lambda)=\sum_{i=1}^{n}\bigg(1.3062974*10^8\dfrac{f_i}{10^{7}/6199.2} \text{exp}\bigg[-\bigg(\dfrac{1/\lambda-1/\lambda_i}{1/6199.2}\bigg)^2\bigg]\bigg) \tag{3}
$$

In Equation 3, your wavelengths ($\lambda$) are the x-values and $\lambda_i$ is the computed wavelength corresponding to the transition with oscillator strength $f_i$.

**Below is the Python form of Equation 2. You will use this function to broaden your computed peaks.**

In [ ]:
# @title Define the broaden_peaks_gaussian function

def broaden_peak_gaussian(X,h,f,sigma=1/3099.6/2.5):
  '''
  X: wavelength range
  h: individual peak wavelength
  f: oscillator strength
  sigma: broadening parameter (as factors of 1/3099.6/x where you want to change x as needed)
  '''

  f2=1e7*sigma
  A=1.3062974*10**8
  e1=1/h-1/X

  return A*(f/f2)*np.exp(-(e1/sigma)**2)

In [ ]:
# @title Interactive Plot
# @markdown Hit the play button once to start the interactive plot
%run Broadening.py

In [ ]:
# @title **Question 1**
%%html
 <head>
    <!-- See http://docs.mathjax.org/en/latest/web/start.html#using-mathjax-from-a-content-delivery-network-cdn -->
    <script type="text/javascript" id="MathJax-script" async
        src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js">
    </script>
  </head>

<style>
div.gold-note {
    color: #796748; /* Dark gold for text */
    background-color: #F7F0E2; /* Light cream/gold background */
    border-left: 5px solid #D8B266; /* Medium gold accent border */
    padding: 0.5em;
    font-size: 1.25em;
    line-height: 1.5;
    font-family: Arial, sans-serif;
}
div.gold-note ul {
    margin: 0.5em 0; /* Space around the list */
}
div.gold-note li {
    margin-bottom: 0.5em; /* Space between list items */
}
</style>

<div class="gold-note">
    <p> <strong> Question: </strong> Explain how the quantized transition energies correspond to peaks in the spectra.</p>
</div>


---

**ANSWER GOES HERE**

---

## 1.3 Plot the combined experimental and computed spectra


In [ ]:
# @title Note: Matplotlib Customization
%%html
 <head>
    <!-- See http://docs.mathjax.org/en/latest/web/start.html#using-mathjax-from-a-content-delivery-network-cdn -->
    <script type="text/javascript" id="MathJax-script" async
        src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js">
    </script>
  </head>

<style>
div.purple-box {
    color: #4b0082; /* Indigo for text */
    background-color: #f3e5f5; /* Light lavender background */
    border-left: 5px solid #7b1fa2; /* Medium purple border */
    padding: 0.5em;
    font-size: 1.25em; /* Matches the surrounding text size */
    line-height: 1.5; /* Ensures readability */
    font-family: Arial, sans-serif; /* Clean, modern font */
}
div.purple-box ul {
    margin: 0.5em 0; /* Space around the list */
}
div.purple-box li {
    margin-bottom: 0.5em; /* Space between list items */
}
</style>

<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Display Python Code</title>
    <style>
        code {
            background-color: #f4f4f4;
            padding: 5px 10px;
            display: block;
            border-left: 4px solid #333;
            font-family: Consolas, monospace;
            white-space: pre;
        }
    </style>
</head>

<div class="purple-box">

<strong> Note: </strong>

<h3>Importing inset figures</h3>
<p>Sometimes, you don't want to use PowerPoint or other software to
combine images together because of loss of resolution, or the extra time to make sure everything is correct.
So, there are packages to create offset images that we can use to import images into a matplotlib figure.

<code># Packages to inset figures into plots (already loaded)
from matplotlib.offsetbox import TextArea, DrawingArea, OffsetImage, AnnotationBbox
import matplotlib.image as mpimg
</code>
<p></p>
<p><h3>Custom Legends</h3>
<p>Sometimes, you need a custom legend beyond what labels in a plt.plot() provides.
To create a custom legend, the following packages can be imported.</p>

<code># Create lines for custom legends (already imported in this notebook)
import matplotlib.lines as mlines
from matplotlib.lines import Line2D
</code>

</html>

</div>

### Custom Legends

The code block below can be used to create custom legends that are independent of the labels given inside a <code>plt.plot()</code> command.

```python
# Copy this block here and edit accordingly for a custom legend
custom_lines = [Line2D([0], [0], color='C0', lw=3),
                Line2D([0], [0], color='k', lw=3,)]
ax1.legend(custom_lines, ['Computed', 'Experiment'],fontsize=20)
```

In the code block above, the 'C0' is the default blue color and is used to label the 'Computed' spectrum whereas the 'k' is black and is used to label the 'Experimental' spectrum.

In [ ]:
# @title Exercise
%%html
 <head>
    <!-- See http://docs.mathjax.org/en/latest/web/start.html#using-mathjax-from-a-content-delivery-network-cdn -->
    <script type="text/javascript" id="MathJax-script" async
        src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js">
    </script>
  </head>


<style>
div.orange-alert {
    color: #854f00; /* Darker shade of orange for text */
    background-color: #ffe6cc; /* Light orange background */
    border-left: 5px solid #ff9933; /* Bright orange border */
    padding: 0.5em;
    font-size: 1.25em; /* Matches the surrounding text size */
    line-height: 1.5; /* Ensures readability */
}
div.orange-alert ul {
    margin: 0.5em 0; /* Space around the list */
}
div.orange-alert li {
    margin-bottom: 0.5em; /* Space between list items */
}
</style>

<div class="orange-alert">
<p> <strong> Exercise: </strong> </p>
<ul>
  <li> Combine everything (custom legends, loops, broaden_peak_gaussian, etc.)
    to plot a combined figure of the computed and experimental spectra (see below).</li>
  <li> Create a two-axis plot to simultaneously plot the computed and experimental spectra on the same image.</li>
  <li> Either use a for loop or a function to repeat the task for all three dyes</li>
</ul>
<div>
  <center> <img height="400" src="https://github.com/pmpatel-udallas/CHE-3131/blob/main/Rotation-4-Quantum-Theory/Figures/Fig6_UVvis.png?raw=True"> </center>
</div>
</div>


In [ ]:
# Define the linear space for your computed spectra (match the experimental range)
xmin=         # Min wavelength
xmax=         # Max wavelength

X=

# Start off with a list of zeros.
comp_spec_dye1=
comp_spec_dye2=
comp_spec_dye3=

# Create the for loop to loop through all twenty peaks with the broaden_peak_gaussian function
# You will use the += operation to add the contributions of each computed peak



In [ ]:
# Use a twin axis to plot the computed spectrum on one axis and the experimental spectrum on a twin second axis
# This allows for qualitative comparisons since intensities are relative.

fig, ax1 = plt.subplots(figsize=(12,8))
ax2 = ax1.twinx() # Create a dual-axis plot


# Plot the computed spectra (data) and the vertical lines (transitions)
# Shift the x values by the difference between the computed and experimental peaks



# Plot the experimental UV-Vis data


# Grabs the default axes limits
ymin1,ymax1=ax1.get_ylim()
ymin2,ymax2=ax2.get_ylim()

# Set the bounds on each axis by changing the values inside the parentheses
ax1.set_ylim(ymin1,ymax2)
ax2.set_ylim(ymin2,ymax2)

ax1.set_xlim(xmin,xmax)
# Label the axes in a subplot using the Axes methods set_xlabel() and set_ylabel()
ax1.set_xlabel('X title')
ax1.set_ylabel('Y title 1')
ax2.set_ylabel('Y title 2')

plt.tight_layout()

In [ ]:
# Repeat this for dyes 2 and 3 so you have figures for all the dyes



### Inset Figures

The code below is a sample of how to create inset images within Python. Sometimes, you don't want to use PowerPoint or other software to combine images together because of loss of resolution, or the extra time to make sure everything is correct. So, there are packages to create offset images that we can use to import images into a matplotlib figure.

```python
# Load an image (PNG/JPG/etc.)
img = mpimg.imread("/content/...")   # path to your image

fig, ax = plt.subplots(figsize=(5, 4))

# Plot your data
ax.plot([0, 1, 2, 3], [0, 2, 1, 3])

# Create the inset image object
imagebox = OffsetImage(img, zoom=0.1)   # zoom adjusts size

# Create an annotation box to place it
ab = AnnotationBbox(
    imagebox,
    (1.0, 3.0),       # (x, y) position in data coordinates
    frameon=False,    # If True, draws a box around the inset
)

# Add to the axes
ax.add_artist(ab)

plt.show()

In [ ]:
# Play around with inset figures with your orbital pictures if you want



# Part 2. Finite Well

The finite well model extends the 1D infinite potential well by introducing finite barriers, allowing for quantum tunneling and wavefunctions that decay into the barrier regions. This results in lower, more closely spaced energy levels and offers a more realistic approximation of electron behavior in molecules. Computational chemistry techniques reinforce this connection by visualizing MOs of the 3D geometry, which leads to an estimation of the effective box length and well depth. Rendering MOs in the context of a finite well also helps illustrate how the well depth limits the number of bound states, highlighting where the particle-in-a-box approximation breaks down and higher energy orbitals begin to resemble unbound or delocalized states. **By combining these computational insights with the finite well model, this notebook will demonstrate how to use the finite well approximation as a predictive tool for absorption spectra.**

<div>
  <center> <img height="500" src="https://github.com/pmpatel-udallas/CHE-3131/blob/main/Rotation-4-Quantum-Theory/Figures/Fig3_finitewell.png?raw=True"> </center>
</div>

Consider a particle in a finite well model where
\begin{align}
	k_0 &= \sqrt{\dfrac{2m_e}{\hbar}(V_0-E)}\\
	k_1 &= \sqrt{\dfrac{2m_eE}{\hbar^2}}
\end{align}
To determine the energy eigenvalues from the wavefunction, we begin the derivation where $\Psi_\text{I}(x)$ and $\Psi_\text{II}(x)$ must follow the boundary conditions at x=0, i.e., $\Psi_\text{I}(0)=\Psi_\text{II}(0)$ and $\Psi'_\text{I}(0)=\Psi'_\text{II}(0)$.

\begin{align}
	Ae^{k_0(0)} &= Bsin(k_1*0)+Ccos(k_1*0) \tag{4}\\
	Ak_0e^{k_0(0)} &= Bk_1cos(k_1*0)-Ck_1sin(k_1*0) \tag{5}
\end{align}

By solving the system of equations above,

\begin{align}
	C&=A \tag{6}\\
	B&=A\frac{k_0}{k_1} \tag{7}
\end{align}

At x=L, the system of equations containing $\Psi(L)$ and $\Psi'(L)$ is

\begin{align}
	De^{-k_0L} &= Bsin(k_1L)+Ccos(k_1L) \tag{8}\\
	De^{-k_0L} &= Bk_1cos(k_1L)-Ck_1sin(k_1L)\tag{9}
\end{align}

By algebra,

\begin{align}
	Dk_0e^{-k_0L} &= A\dfrac{k_0^2}{k_1}sin(k_1L)+Ak_0cos(k_1L) \tag{10}\\
	Dk_0e^{-k_0L} &= -A\dfrac{k_0}{k_1}k_1cos(k_1L)+Ak_1sin(k_1L)) \tag{11}
\end{align}

By equality, the above equations may be solved for the quantity $Dk_0e^{-k_0L}$

\begin{align}
	\dfrac{k_0^2}{k_1}sin(k_1L)+k_0cos(k_1L) &= -k_0cos(k_1L)+k_1sin(k_1L) \\
	k_0^2sin(k_1L)+k_1k_0cos(k_1L) &= -k_1k_0cos(k_1L)+k_1^2sin(k_1L)\\
	2k_1k_0cos(k_1L) &= (k_1^2-k_0^2)sin(k_1L)
\end{align}

\begin{equation}
	2k_1k_0 = (k_1^2-k_0^2)tan(k_1L) \tag{12}
\end{equation}

By substituting $k_0$ and $k_1$ into the above equation,

\begin{equation}
	2\bigg(\dfrac{2m_eE}{\hbar^2}\bigg)\sqrt{E(V_0-E)} = \bigg(\dfrac{2m_eE}{\hbar^2}-\dfrac{2m_e(V_0-E)}{\hbar^2}\bigg)tan\bigg(\sqrt{\dfrac{2m_eEL^2}{\hbar^2}}\bigg)
\end{equation}

\begin{equation}
	\dfrac{2\sqrt{E(V_0-E)}}{2E-V_0} = tan\bigg(\sqrt{\dfrac{2m_eEL^2}{\hbar^2}}\bigg)\tag{13}
\end{equation}

Solving Equation 13 for $E$ correspond to the energy levels of the finite well. You will plot these functions here to solve for all values of $E$.

For this module, let
\begin{align}
f(x) &= \dfrac{2\sqrt{E(V_0-E)}}{2E-V_0}\\
g(x) &= tan\bigg(\sqrt{\dfrac{2m_eEL^2}{\hbar^2}}\bigg)
\end{align}

## 2.1 Set up the model

In [ ]:
# @title Exercise
%%html
 <head>
    <!-- See http://docs.mathjax.org/en/latest/web/start.html#using-mathjax-from-a-content-delivery-network-cdn -->
    <script type="text/javascript" id="MathJax-script" async
        src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js">
    </script>
  </head>


<style>
div.orange-alert {
    color: #854f00; /* Darker shade of orange for text */
    background-color: #ffe6cc; /* Light orange background */
    border-left: 5px solid #ff9933; /* Bright orange border */
    padding: 0.5em;
    font-size: 1.25em; /* Matches the surrounding text size */
    line-height: 1.5; /* Ensures readability */
}
div.orange-alert ul {
    margin: 0.5em 0; /* Space around the list */
}
div.orange-alert li {
    margin-bottom: 0.5em; /* Space between list items */
}
</style>

<div class="orange-alert">
<p> <strong> Exercise: </strong> </p>
<ul>
  <li> Fill in the blanks in the following code blocks to construct the functions used to compute the finite well.</li>
</ul>

</div>


### Define the constants and conversion factors

In [ ]:
eV2J= # Convert from eV to J (4 sig figs)
h=6.62606957e-34 # Planck's constant (J·s)
hbar=  # ℏ in J·s
me= # mass of electron in kg (3 sig figs)

### Define the functions

In [ ]:
def f(Ei,V0):
  '''Ei is in eV'''
  '''V0 is in eV'''
  # Convert from eV to J
  E= Ei*eV2J
  V= V0*eV2J

  # Suppress RuntimeWarning for invalid value encountered in sqrt
  with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    #----------------------
    # Numerator
    a= # CODE THIS (do not change indentation)
    #----------------------

  #----------------------
  # Denominator
  b=   # CODE THIS
  #----------------------

  # Return the value of a divided by b
  return # CODE THIS

def g(Ei,L):
  '''Ei is in eV'''
  '''L is in m'''

  # Convert from eV to J
  E= Ei*eV2J

  #----------------------
  # Define terms inside the square root term of g(x)
  a= #CODE THIS
  b= #CODE THIS
  #----------------------

  # Suppress RuntimeWarning for invalid value encountered in sqrt
  with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)

    #----------------------
    # Square Root function
    c= # CODE THIS
    #----------------------

  # Return the tangent (np.tan) of the square root term defined as c
  return # CODE THIS

# Define the difference function between f(x) and g(x) to help find roots
def j(x, V0_local, L_local):
  return f(x,V0_local)-g(x,L_local)

# Define the intersections
def intersections(X2,Y2,Z2):
  roots_idx = np.argwhere(np.diff(np.sign(Y2-Z2)) != 0).flatten()
  Xint = X2[roots_idx]
  return Xint[1::2]

## 2.2 Play with the model

In [ ]:
# @title Interactive Plot
# @markdown Hit the play button once to start the interactive plot
%run FiniteWell.py

interactive(children=(FloatSlider(value=10.0, description='V₀ (eV)', max=30.0, min=8.0, step=0.01), FloatSlide…

<Figure size 640x480 with 0 Axes>

In [ ]:
# @title **Question 2**
%%html
 <head>
    <!-- See http://docs.mathjax.org/en/latest/web/start.html#using-mathjax-from-a-content-delivery-network-cdn -->
    <script type="text/javascript" id="MathJax-script" async
        src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js">
    </script>
  </head>

<style>
div.gold-note {
    color: #796748; /* Dark gold for text */
    background-color: #F7F0E2; /* Light cream/gold background */
    border-left: 5px solid #D8B266; /* Medium gold accent border */
    padding: 0.5em;
    font-size: 1.25em;
    line-height: 1.5;
    font-family: Arial, sans-serif;
}
div.gold-note ul {
    margin: 0.5em 0; /* Space around the list */
}
div.gold-note li {
    margin-bottom: 0.5em; /* Space between list items */
}
</style>

<div class="gold-note">
    <p> <strong> Question: </strong> What happens as you change V0 and L?</p>
</div>


---

**ANSWER GOES HERE**

---

## 2.3 Finding $V_0$ and L

(**You already did this**)

### Determining L
The length of the box will be the Euclidean distance between the $C$ atoms outside of the $N$ atoms in the chain. Take the chain between the $N$ atoms and add the carbons in the ring that form a linear chain. To do this:

1.   You will need to open the molecules in Chemcraft.
2.   Get the distance between the N atoms by selecting the two N atoms.
3.   Then select the C atoms on the outer edge of the N atoms; this distance will increase
4.   Then deselect the N atoms to get the distance between the selected C atoms
5.   This distance is the box length for your calculations.

### Determining $V_0$
You will need the range of orbitals from HOMO-$n_1$ to LUMO+$n_2$ (where $n_1$ and $n_2$ are different orbital numbers relative to HOMO and LUMO) to calculate the well depth $V_0$. To do this:
1. Render the orbitals in Chemcraft.
2. Identify the appropriate orbitals that represent the correct nodal structures of the conjugated $\pi$ system.
3. Mark down the orbital energies of the identified HOMO-$n_1$ and LUMO+2 orbitals. The difference in energies is the first guess at the orbital energies. You may have to add 0.5-1 eV to this as a buffer (rounding also works).

In [ ]:
# Enter your calculate box lengths in A
L1=
L2=
L3=

# Enter your calculated potentials here
V1=
V2=
V3=

## 2.4 Plotting the finite well energies

In [ ]:
# @title Note: Solving the Intersections
%%html
 <head>
    <!-- See http://docs.mathjax.org/en/latest/web/start.html#using-mathjax-from-a-content-delivery-network-cdn -->
    <script type="text/javascript" id="MathJax-script" async
        src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js">
    </script>
  </head>

<style>
div.purple-box {
    color: #4b0082; /* Indigo for text */
    background-color: #f3e5f5; /* Light lavender background */
    border-left: 5px solid #7b1fa2; /* Medium purple border */
    padding: 0.5em;
    font-size: 1.25em; /* Matches the surrounding text size */
    line-height: 1.5; /* Ensures readability */
    font-family: Arial, sans-serif; /* Clean, modern font */
}
div.purple-box ul {
    margin: 0.5em 0; /* Space around the list */
}
div.purple-box li {
    margin-bottom: 0.5em; /* Space between list items */
}
</style>

<div class="purple-box">

<strong> Note: Solving the intersections</strong>
<p>
The fsolve function in Python is part of the scipy.optimize module
and is used to find the roots of a system of nonlinear equations.
It numerically solves the equation \( f(x)=0\) where \(f(x)\) is a callable function.
</div>

In [ ]:
def finite_well(V0,LA):
  '''V0 is in eV'''
  '''LA is in Angstrom'''

  L = LA*1e-10 #m

  # Define the space to define the function
  X = np.linspace(0.005,V0, 10000)

  # This is included to remove warnings
  with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    Y = f(X,V0)
    Z = g(X,L)

  # Filter the data to remove values approaching the asymptotes
  valid_indices = ~np.isnan(Y) & ~np.isnan(Z) & np.isfinite(Y) & np.isfinite(Z) & (np.abs(Z) < V0)
  X2=X[valid_indices]
  Y2=Y[valid_indices]
  Z2=Z[valid_indices]

  # Solve for the roots
  with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning) # Ignore RuntimeWarning for fsolve
    Xint = fsolve(lambda x_val: j(x_val, V0, L), intersections(X2,Y2,Z2))

  for i in range(len(Xint)):
    print('E_'+str(i+1)+' =','%0.3f'%Xint[i])

  # Return the roots as the final answer
  return Xint

In [ ]:
# @title **Question 3**
%%html
 <head>
    <!-- See http://docs.mathjax.org/en/latest/web/start.html#using-mathjax-from-a-content-delivery-network-cdn -->
    <script type="text/javascript" id="MathJax-script" async
        src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js">
    </script>
  </head>

<style>
div.gold-note {
    color: #796748; /* Dark gold for text */
    background-color: #F7F0E2; /* Light cream/gold background */
    border-left: 5px solid #D8B266; /* Medium gold accent border */
    padding: 0.5em;
    font-size: 1.25em;
    line-height: 1.5;
    font-family: Arial, sans-serif;
}
div.gold-note ul {
    margin: 0.5em 0; /* Space around the list */
}
div.gold-note li {
    margin-bottom: 0.5em; /* Space between list items */
}
</style>

<div class="gold-note">
    <p> <strong> Question: </strong> The code block for the <code>finite_well()</code> function does the same thing as the interactive plot without the visual aspect.
    What is the result of the for loop in lines 27-28 of the cell above?</p>
</div>


---

**ANSWER GOES HERE**

---

## 2.5 Calculating $\lambda_{max}$

In [ ]:
# @title Exercise
%%html
 <head>
    <!-- See http://docs.mathjax.org/en/latest/web/start.html#using-mathjax-from-a-content-delivery-network-cdn -->
    <script type="text/javascript" id="MathJax-script" async
        src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js">
    </script>
  </head>

<style>
div.orange-alert {
    color: #854f00; /* Darker shade of orange for text */
    background-color: #ffe6cc; /* Light orange background */
    border-left: 5px solid #ff9933; /* Bright orange border */
    padding: 0.5em;
    font-size: 1.25em; /* Matches the surrounding text size */
    line-height: 1.5; /* Ensures readability */
}
div.orange-alert ul {
    margin: 0.5em 0; /* Space around the list */
}
div.orange-alert li {
    margin-bottom: 0.5em; /* Space between list items */
}
</style>

<div class="orange-alert">
<p> <strong> Exercise: </strong> </p>
<ul>
  <li> Use your values for \(V_0\) and \(L\) to compute the respective energy levels for the finite well.</li>
  <li> Then, use the appropriate energy levels to calculate the theoretical \( \lambda_{max}\) for the finite well models.</li>
</ul>

</div>

In [ ]:
en_1=
print(en_1)
print('λ_max for dye 1:','%0.2f'%())

In [ ]:
en_2=
print(en_2)
print('λ_max for dye 2:','%0.2f'%())

In [ ]:
en_3=
print(en_3)
print('λ_max for dye 3:','%0.2f'%())

In [1]:
# @title Key Points
%%html
 <head>
    <!-- See http://docs.mathjax.org/en/latest/web/start.html#using-mathjax-from-a-content-delivery-network-cdn -->
    <script type="text/javascript" id="MathJax-script" async
        src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js">
    </script>
  </head>


<style>
div.green-note {
    color: #155724; /* Dark green for text */
    background-color: #d4edda; /* Light green background */
    border-left: 5px solid #28a745; /* Bright green border */
    padding: 0.5em;
    font-size: 1.25em; /* Consistent with text size */
    line-height: 1.5; /* Ensures readability */
    font-family: Arial, sans-serif; /* Clean and modern font */
}
div.green-note ul {
    margin: 0.5em 0; /* Space around the list */
}
div.green-note li {
    margin-bottom: 0.5em; /* Space between list items */
}
</style>

<div class="green-note">
    <strong>Key Points:</strong>
    <ul>
      <li> Use Python functions and loops to save time in doing repetitive tasks</li>
      <li> Use subplots and a twin axis to plot a computed and experimental UV-vis spectra on the same figure </li>
      <li> Compute energy levels in a finite well model</li>
</div>